# AI Agent Foundations

This notebook demonstrates the foundational components for building AI agents:
1. Environment Setup
2. Basic OpenAI SDK usage
3. Chain Calls (Generate -> Evaluate)
4. Process-Evaluate Agent Loops

In [ ]:
# 1. Environment Setup
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
openai_api_base = os.getenv('OPENAI_API_BASE')

openai = OpenAI(
    api_key=openai_api_key,
    base_url=openai_api_base
)

print("Environment setup complete. OpenAI client initialized.")

In [ ]:
# 2. Example of calling OpenAI SDK

def simple_chat(prompt, model="gpt-4o", temperature=0.7):
    """
    A simple wrapper to call OpenAI ChatCompletion.
    """
    try:
        response = openai.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
            temperature=temperature
        )
        return response.choices[0].message.content
    except Exception as e:
        print(f"Error calling OpenAI: {e}")
        return None

# Example usage:
# response = simple_chat("Hello, how are you today?")
# print(response)

In [ ]:
# 3. Example of Chain Call (Generate -> Evaluate)

def generate_content(topic):
    """Stub function to generate content using LLM"""
    prompt = f"Write a short 2-sentence story about {topic}."
    return simple_chat(prompt)

def evaluate_content(content):
    """Stub function to evaluate the generated content"""
    prompt = f"Evaluate the following story for creativity and grammar on a scale of 1-10:\n\n{content}"
    return simple_chat(prompt)

def run_chain_execution(topic):
    print(f"--- Starting Chain for topic: {topic} ---")
    
    # Step 1: Generate
    print("Generating content...")
    content = generate_content(topic)
    print(f"Generated Content:\n{content}\n")
    
    # Step 2: Evaluate
    if content:
        print("Evaluating content...")
        evaluation = evaluate_content(content)
        print(f"Evaluation Result:\n{evaluation}")
    else:
        print("Generation failed, skipping evaluation.")

# Example usage:
# run_chain_execution("a flying cat")

In [ ]:
# 4. Example of Process-Evaluate Agent Loop

def check_completion(result, goal):
    """
    Uses LLM to evaluate if the result meets the goal.
    Return (bool, feedback)
    """
    prompt = f"Goal: {goal}\nResult: {result}\n\nHas the goal been fully achieved? Respond with 'YES' or 'NO' followed by a short explanation."
    response = simple_chat(prompt)
    
    is_complete = response and response.strip().upper().startswith("YES")
    return is_complete, response

def refine_result(previous_result, feedback):
    """
    Uses LLM to improve the result based on feedback.
    """
    prompt = f"Original: {previous_result}\nFeedback: {feedback}\n\nPlease improve the original text based on the feedback."
    return simple_chat(prompt)

def agent_loop(goal, max_iterations=3):
    print(f"--- Starting Agent Loop for Goal: {goal} ---")
    
    # Initial attempt
    current_result = simple_chat(f"Attempt to achieve this goal: {goal}")
    print(f"Initial Result: {current_result}\n")
    
    for i in range(max_iterations):
        print(f"Iteration {i+1}: Evaluating...")
        success, feedback = check_completion(current_result, goal)
        
        print(f"Feedback: {feedback}")
        
        if success:
            print("\nGoal Achieved! Exiting loop.")
            break
        
        print("Refining result...")
        current_result = refine_result(current_result, feedback)
        print(f"Refined Result: {current_result}\n")
    else:
        print("Max iterations reached. Stopping.")
    
    return current_result

# Example usage:
# final_output = agent_loop("Write a haiku about rust programming")